In [ ]:
import pandas as pd

In [ ]:
import numpy as np

def gabriel_eigen_impute(X, tol=1e-4, max_iter=100, m=None):
    """
    Perform imputation on the matrix X using the regularized GabrielEigen method.

    Parameters:
    - X: 2D numpy array with missing values as np.nan
    - tol: Convergence tolerance
    - max_iter: Maximum number of iterations
    - m: Rank of the approximation (if None, it will be determined automatically)

    Returns:
    - X_imputed: Imputed matrix
    """
    X = np.array(X, dtype=float)  # Ensure X is a NumPy array
    n, p = X.shape

    # Step 1: Replace missing values with column means
    col_means = np.nanmean(X, axis=0)
    inds = np.where(np.isnan(X))
    X[inds] = np.take(col_means, inds[1])

    # Initialize variables for convergence check
    X_prev = np.copy(X)
    converged = False
    iter_count = 0

    while not converged and iter_count < max_iter:
        # Step 2: Standardize the columns
        col_means = np.mean(X, axis=0)
        col_stds = np.std(X, axis=0, ddof=1)
        X_std = (X - col_means) / col_stds

        # Handle zero standard deviation
        col_stds[col_stds == 0] = 1
        X_std = (X - col_means) / col_stds

        # Step 3: Perform regularized SVD
        U, D, Vt = reg_svd(X_std, m)

        # Reconstruct the matrix using rank-m approximation
        X_hat_std = np.dot(U, np.dot(np.diag(D), Vt))

        # Step 5: De-standardize the imputed values
        X_hat = X_hat_std * col_stds + col_means

        # Only update missing values
        X[inds] = X_hat[inds]

        # Check for convergence
        diff = np.linalg.norm(X - X_prev) / np.linalg.norm(X_prev)
        if diff < tol:
            converged = True
        X_prev = np.copy(X)
        iter_count += 1

    return X

def reg_svd(X_std, m=None):
    """
    Compute the regularized SVD of X_std.

    Parameters:
    - X_std: Standardized matrix
    - m: Desired rank (if None, it will be determined based on explained variance)

    Returns:
    - U: Left singular vectors
    - D: Singular values
    - Vt: Right singular vectors (transposed)
    """
    n, p = X_std.shape
    k = min(n, p)

    # Step 4: Choose lambda via direct search
    lambdas = np.arange(0, 1.1, 0.1)
    min_f = np.inf
    best_lambda = 0
    best_U = None
    best_V = None

    for lmbda in lambdas:
        U, V, f = compute_reg_svd(X_std, k, lmbda)
        if f < min_f:
            min_f = f
            best_lambda = lmbda
            best_U = U
            best_V = V

    # Compute final SVD on U * V.T
    UVt = np.dot(best_U, best_V.T)
    U_final, D, Vt_final = np.linalg.svd(UVt, full_matrices=False)

    # Determine rank m if not specified
    if m is None:
        cumulative_energy = np.cumsum(D**2)
        total_energy = cumulative_energy[-1]
        m = np.searchsorted(cumulative_energy, 0.8 * total_energy) + 1

    # Keep only the first m components
    U_final = U_final[:, :m]
    D = D[:m]
    Vt_final = Vt_final[:m, :]

    return U_final, D, Vt_final

def compute_reg_svd(X, k, lmbda, tol=1e-4, max_iter=100):
    """
    Compute the regularized SVD for a given lambda.

    Parameters:
    - X: Standardized matrix
    - k: Desired rank
    - lmbda: Regularization parameter
    - tol: Convergence tolerance
    - max_iter: Maximum number of iterations

    Returns:
    - U: Left singular vectors
    - V: Right singular vectors
    - f: Objective function value
    """
    n, p = X.shape
    V = np.random.rand(p, k)
    U = np.zeros((n, k))
    f_prev = np.inf

    I_k = np.eye(k)

    for _ in range(max_iter):
        # Update U
        VVt = np.dot(V.T, V) + lmbda * I_k
        VVt_inv = np.linalg.pinv(VVt)
        U = np.dot(np.dot(X, V), VVt_inv)

        # Update V
        UUt = np.dot(U.T, U) + lmbda * I_k
        UUt_inv = np.linalg.pinv(UUt)
        V = np.dot(np.dot(X.T, U), UUt_inv)

        # Compute objective function
        UVt = np.dot(U, V.T)
        residual = X - UVt
        f = np.linalg.norm(residual, 'fro')**2 + lmbda * (np.linalg.norm(U, 'fro')**2 + np.linalg.norm(V, 'fro')**2)

        # Check for convergence
        if abs(f_prev - f) < tol:
            break
        f_prev = f

    return U, V, f


In [ ]:
# Example usage with a matrix containing missing values
X = np.array([
    [1.0, np.nan, 3.0],
    [4.0, 5.0, np.nan],
    [7.0, 8.0, 9.0]
])

X_imputed = gabriel_eigen_impute(X)
print("Imputed Matrix:")
print(X_imputed)


In [ ]:
example_dataset_path = '../../datasets/processados/tratados/10-06-2023/bbr/tratado bbr esmond data mg-rs 06-10-2023.csv'

In [ ]:
import pandas as pd
import numpy as np

def df_transformation(path):
    df = pd.read_csv(path)
    df['Data'] = pd.to_datetime(df['Data'].str.strip(), format='%d-%m-%Y')
    df['Timestamp'] = pd.to_datetime(df['Data'].dt.strftime('%Y-%m-%d') + ' ' + df['Intervalo'].str.split(' a ').str[0])
    df['UnixTimestamp'] = df['Timestamp'].apply(lambda x: int(x.timestamp()))
    new_df = pd.DataFrame()
    new_df['Timestamp'] = df['UnixTimestamp']
    new_df['Vazao'] = df['Vazao']

    return new_df    

def df_to_matrix_transformation(df, columns_quantity = 0):
    df['Vazao'] = df['Vazao'].replace(-1, np.nan)
    matrix = df.to_numpy()
    return matrix


In [ ]:
new_df = df_transformation(example_dataset_path)
matrix = df_to_matrix_transformation(new_df)

In [ ]:
gabriel_eigen_impute(matrix)